# 3 — Construcción de Gold

Gold publica tres productos con granos explícitos: ventas diarias, riesgo por cliente y resumen operativo por lote. Se reconstruyen desde Silver para que una reejecución sea determinista.

In [0]:
dbutils.widgets.text("student_id", "alumno01")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"])
dbutils.widgets.text("expected_batch_id", "batch_002")
dbutils.widgets.text("job_run_id", "interactive")

In [0]:
%run ../common/config

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
tx = qualified_table(config, "silver_transactions")
customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
quarantine = qualified_table(config, "silver_transactions_quarantine")
for name in [tx, customers, products, quarantine]:
    assert spark.catalog.tableExists(name), f"Falta {name}. Ejecutá la tarea 02."

In [0]:
daily = qualified_table(config, "gold_daily_sales")
spark.sql(f"""
CREATE OR REPLACE TABLE {daily} USING DELTA AS
SELECT to_date(t.event_ts) sale_date, c.country, p.category, t.payment_channel,
       count(*) transaction_count, count(DISTINCT t.customer_id) unique_customers,
       cast(sum(t.amount) AS DECIMAL(18,2)) total_amount,
       cast(avg(t.amount) AS DECIMAL(18,2)) average_amount,
       sum(t.is_fraud) fraud_transactions,
       cast(sum(t.is_fraud) / count(*) AS DECIMAL(8,4)) fraud_rate
FROM {tx} t
JOIN {customers} c ON t.customer_id = c.customer_id
JOIN {products} p ON t.product_id = p.product_id
GROUP BY to_date(t.event_ts), c.country, p.category, t.payment_channel
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
risk = qualified_table(config, "gold_customer_risk")
spark.sql(f"""
CREATE OR REPLACE TABLE {risk} USING DELTA AS
WITH metrics AS (
  SELECT t.customer_id, c.country, c.segment, count(*) transaction_count,
         cast(sum(t.amount) AS DECIMAL(18,2)) total_amount,
         cast(avg(t.amount) AS DECIMAL(18,2)) average_amount,
         count(DISTINCT t.device_id) distinct_devices, sum(t.is_fraud) fraud_transactions,
         max(t.event_ts) last_transaction_at
  FROM {tx} t JOIN {customers} c ON t.customer_id = c.customer_id
  GROUP BY t.customer_id, c.country, c.segment
)
SELECT *, CASE WHEN fraud_transactions >= 2 THEN 'high'
               WHEN fraud_transactions = 1 OR average_amount >= 1500 THEN 'medium'
               ELSE 'low' END risk_level
FROM metrics
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
batch = qualified_table(config, "gold_batch_summary")
spark.sql(f"""
CREATE OR REPLACE TABLE {batch} USING DELTA AS
WITH accepted AS (
  SELECT source_batch_id, count(*) accepted_transactions,
         cast(sum(amount) AS DECIMAL(18,2)) total_amount, sum(is_fraud) fraud_transactions
  FROM {tx} GROUP BY source_batch_id
), rejected AS (
  SELECT source_batch_id, count(*) rejected_transactions FROM {quarantine} GROUP BY source_batch_id
)
SELECT coalesce(a.source_batch_id, r.source_batch_id) source_batch_id,
       coalesce(a.accepted_transactions, 0) accepted_transactions,
       coalesce(r.rejected_transactions, 0) rejected_transactions,
       coalesce(a.total_amount, cast(0 AS DECIMAL(18,2))) total_amount,
       coalesce(a.fraud_transactions, 0) fraud_transactions, current_timestamp() refreshed_at
FROM accepted a FULL OUTER JOIN rejected r ON a.source_batch_id = r.source_batch_id
""")
display(spark.table(batch).orderBy("source_batch_id"))
display(spark.table(daily).orderBy("sale_date", "country").limit(20))

source_batch_id,accepted_transactions,rejected_transactions,total_amount,fraud_transactions,refreshed_at
batch_002,21,2,1771.14,0,2026-10-09T03:19:40.916Z
batch_003,21,2,1792.35,0,2026-10-09T03:19:40.916Z
batch_004,21,2,1813.56,0,2026-10-09T03:19:40.916Z
batch_005,200,2,143919.00,26,2026-10-09T03:19:40.916Z
batch_006,200,2,144121.00,26,2026-10-09T03:19:40.916Z
batch_007,200,2,144323.00,26,2026-10-09T03:19:40.916Z
batch_008,200,2,144525.00,26,2026-10-09T03:19:40.916Z
batch_009,200,2,144727.00,27,2026-10-09T03:19:40.916Z
batch_010,201,2,146928.99,28,2026-10-09T03:19:40.916Z
initial,49882,51,50068396.35,6550,2026-10-09T03:19:40.916Z


sale_date,country,category,payment_channel,transaction_count,unique_customers,total_amount,average_amount,fraud_transactions,fraud_rate
2026-03-01,AR,sports,card,85,84,85019.53,1000.23,10,0.1176
2026-03-01,AR,fashion,wallet,75,74,74658.64,995.45,13,0.1733
2026-03-01,AR,home,wallet,118,111,123693.25,1048.25,14,0.1186
2026-03-01,AR,electronics,wallet,90,86,93980.66,1044.23,13,0.1444
2026-03-01,AR,electronics,transfer,82,77,81687.16,996.18,12,0.1463
2026-03-01,AR,books,wallet,117,111,111285.98,951.16,14,0.1197
2026-03-01,AR,books,transfer,106,100,116692.71,1100.87,17,0.1604
2026-03-01,AR,fashion,card,96,94,84782.98,883.16,10,0.1042
2026-03-01,AR,sports,transfer,102,96,103831.65,1017.96,11,0.1078
2026-03-01,AR,books,card,100,98,111296.79,1112.97,14,0.1400
